In [18]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib


from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [19]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [20]:
df = df.drop("customerID", axis=1)

In [21]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [22]:
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

In [23]:
label_encoders = {}

for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        le = LabelEncoder()

        df[col] = le.fit_transform(
            df[col].astype(str)
        )

        label_encoders[col] = le

In [24]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

### Train-test split

In [25]:
print(df.Churn.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

Churn
0    5174
1    1869
Name: count, dtype: int64


In [26]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42)

In [27]:
param_grid = {
    'n_estimators': [80, 100, 200, 300],
    'max_depth':  [3,5,8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [2, 4] 
}

In [28]:
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)




In [29]:
grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_

In [30]:
print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 300}


In [31]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Random Forest Accuracy:", accuracy)


Random Forest Accuracy: 0.808374733853797


In [32]:
joblib.dump(
    model,
    "random_forest_model.pkl"
)

['random_forest_model.pkl']

In [33]:
print("Model Saved Successfully")

Model Saved Successfully


In [34]:
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)

Training Accuracy: 0.8324458643947462
Testing Accuracy: 0.808374733853797


In [35]:
joblib.dump(
    label_encoders,
    "encoders.pkl"
)

joblib.dump(
    scaler,
    "scaler.pkl"
)

['scaler.pkl']